# 3. Plotly: add interaction to the same figure
**Instructor demonstration.** Begin with the same scatterplot, then add a log scale, country names on hover, and continent groups.

Plotly uses its own figure system. `px.scatter(...)` returns a Plotly `Figure`; Matplotlib methods such as `ax.set_xscale(...)` do not apply to it. `fig.show()` opens the interactive display in the notebook or its configured renderer.


In [ ]:
import pandas as pd
import plotly.express as px

gapminder = pd.read_csv("data/gapminder.csv")
snapshot = gapminder.loc[gapminder["year"] == 2007].copy()


## 1. The simplest scatterplot
The code resembles Seaborn: give it a DataFrame and column names. Hover and zoom already work without extra arguments.


In [ ]:
fig = px.scatter(snapshot, x="gdpPercap", y="lifeExp")
fig.show()


## 2. Change the x scale
**New:** `log_x=True`. Plotly Express exposes this scale choice as an argument to the plotting function.


In [ ]:
fig = px.scatter(snapshot, x="gdpPercap", y="lifeExp", log_x=True)
fig.show()


## 3. Identify observations without labeling every point
**New:** `hover_name="country"`. A reader can inspect a country without covering the chart with text labels. Hover identifies a country; it does not explain why its values differ from another country's.


In [ ]:
fig = px.scatter(snapshot, x="gdpPercap", y="lifeExp", log_x=True, hover_name="country")
fig.show()


## 4. Add continent groups
**New:** `color="continent"`, analogous to Seaborn's `hue="continent"`. Click a legend item to hide or restore a group. Drag to zoom and double-click the plot to reset.


In [ ]:
fig = px.scatter(snapshot, x="gdpPercap", y="lifeExp", log_x=True,
                 hover_name="country", color="continent")
fig.show()


## 5. Supply labels for a reader
**New:** `labels` replaces column names in the display, and `title` identifies the year. These improve interpretation without adding another data mapping.


In [ ]:
fig = px.scatter(snapshot, x="gdpPercap", y="lifeExp", log_x=True,
                 hover_name="country", color="continent",
                 labels={"gdpPercap": "GDP per capita (inflation-adjusted US$)",
                         "lifeExp": "Life expectancy at birth (years)"},
                 title="Countries in 2007")
fig.show()


## 6. Share the interaction as HTML
This file preserves hover, zoom, and legend controls. Including Plotly's JavaScript makes it self-contained and larger. This scatterplot can open offline or be linked from a GitHub Pages project website.

A screenshot loses hover information. Any essential claim should still be clear from visible labels and a caption.


In [ ]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
fig.write_html("outputs/gdp_life_expectancy_2007.html", include_plotlyjs=True)


## Advanced showcase: a moving bubble chart
Bring the intermediate years back with a playable animation and year slider. The GDP-life expectancy coordinates remain familiar; **bubble area represents population, and color identifies country**.

We use 12 explicitly selected countries to keep the country colors and legend readable. They are examples, not a representative sample. Hover supplies exact values. All 12 countries appear in every year.

| Argument | What it contributes |
|---|---|
| `size="pop"` | Population encoded by bubble area |
| `color="country"` | A country's color stays consistent over time |
| `animation_frame="year"` | Play controls and a year slider |
| `animation_group="country"` | Keeps the same country's identity across frames |
| Fixed `range_x` and `range_y` | Keeps the axes from changing as the animation plays |

Press **Play**, then use the year slider to inspect a particular frame. Larger bubbles can obscure smaller ones, and an animation is harder to compare precisely than fixed panels. This is a useful capability to choose when it serves the question.


In [ ]:
selected_countries = ["Australia", "Brazil", "Canada", "China", "France", "India",
                      "Japan", "Nigeria", "South Africa", "Sweden", "United States", "Vietnam"]
selected_history = gapminder.loc[gapminder["country"].isin(selected_countries)].sort_values(["year", "country"])

animation = px.scatter(
    selected_history, x="gdpPercap", y="lifeExp",
    size="pop", color="country", hover_name="country",
    animation_frame="year", animation_group="country",
    log_x=True, range_x=[200, 120000], range_y=[20, 90],
    size_max=55, color_discrete_sequence=px.colors.qualitative.Dark24,
    category_orders={"country": selected_countries},
    hover_data={"pop": ":,.0f", "gdpPercap": ":,.0f", "lifeExp": ":.1f"},
    labels={"gdpPercap": "GDP per capita (inflation-adjusted US$)",
            "lifeExp": "Life expectancy at birth (years)", "pop": "Population (people)"},
    title="GDP, life expectancy, and population: 12 selected countries, 1952-2007",
    template="plotly_white", height=600,
)
animation.show()
animation.write_html("outputs/gapminder_animated.html", include_plotlyjs=True)


**What this shows off:** Plotly coordinates position, population size, country color, hover, and time in one figure. The exported HTML retains the play controls and slider without needing a Streamlit app or a Python server.

The bubble scaling is consistent across frames because the complete selected history is passed to one plotting call. Movement between five-year observations is a visual transition, not a measurement of every intervening year. The chart does not establish why countries changed.

To explore another set, edit `selected_countries`. The full dataset remains available, but assigning a separate color to all 142 countries would make the legend difficult to use.

References: [Plotly animations](https://plotly.com/python/animations/) and [bubble charts](https://plotly.com/python/bubble-charts/).


## How the three libraries differ
| Library | Useful strength | Cost or limitation |
|---|---|---|
| Matplotlib | Precise control of marks, annotations, and layouts | Grouping and statistical summaries often require more code |
| Seaborn | Concise statistical plots and group mappings from DataFrames | Defaults can hide important summary choices; built on Matplotlib's figure system |
| Plotly Express | Hover, zoom, legend controls, and shareable interactive HTML | A separate figure API; advanced customization and interactive exports add complexity |

These strengths overlap. Matplotlib can be interactive, and Plotly can export static figures. Choose based on the question, the audience, and how the result will be shared.


## Where Streamlit fits (about 2 minutes)
**Plotly creates the chart. Streamlit builds a Python web app around charts, tables, controls, and calculations.** Streamlit can display a Plotly figure with `st.plotly_chart(fig)`; it can also display Matplotlib figures.

| Interaction | Who handles it in this example? |
|---|---|
| Hover over a country or zoom the axes | Plotly, in the browser |
| Choose a year with an app dropdown | Streamlit |
| Filter the DataFrame for that year | Python code in the Streamlit app |
| Display the new interactive chart | Streamlit displays a new Plotly figure |

The small **app script** below illustrates the connection. It is reference code, not a notebook cell to run during this lesson. Under the default behavior, changing the year dropdown reruns the script with a new value. Ordinary Plotly hover and zoom do not rerun the Python script.

```python
import pandas as pd
import plotly.express as px
import streamlit as st

gapminder = pd.read_csv("data/gapminder.csv")
year = st.selectbox("Year", sorted(gapminder["year"].unique()))
selected = gapminder.loc[gapminder["year"] == year]
fig = px.scatter(selected, x="gdpPercap", y="lifeExp",
                 log_x=True, hover_name="country", color="continent")
st.plotly_chart(fig)
```

For a single explorable chart, an exported Plotly HTML file may be enough. For controls that run Python, recalculate results, or coordinate several views, Streamlit supplies the surrounding app. A normal Streamlit app needs a running Python service; its script cannot run directly on static GitHub Pages.

No Streamlit installation is required for these notebooks. To try the snippet later, save it as `app.py` in the repository root and run `uv run --with streamlit streamlit run app.py`.

References: [Plotly HTML export](https://plotly.com/python/interactive-html-export/), [Streamlit basics](https://docs.streamlit.io/get-started/fundamentals/main-concepts), [st.plotly_chart](https://docs.streamlit.io/develop/api-reference/charts/st.plotly_chart), and [Streamlit deployment](https://docs.streamlit.io/deploy/concepts).

Student investigations are in [guided practice](practice-1-guided.ipynb). Bubbles, animation, and other extensions are in [extra examples](extra-examples.ipynb).
